# From single neurons to neural networks

**Lecture 12 · Notebook 01 · Core route: 120 minutes · Full laboratory: 180 minutes**

We construct a transparent network with three input signals, two hidden neurons, and two output neurons. We first calculate with individual neurons, derive every backpropagation term, and only then use vectorized layers.

$$
\text{single neuron}\to\text{literal layer}\to\text{pathwise chain rule}
\to\text{matrix backpropagation}\to\text{tested implementation}.
$$

## Learning objectives

### How to use this notebook

You will trace every $z_j^\ell$ and $a_j^\ell$, explain why nonlinear activations are necessary, derive output and hidden errors, translate scalar equations into matrix operations, compare sigmoid–MSE with sigmoid–cross-entropy, check layer gradients, and distinguish what autodifferentiation automates from what scientists must decide.

Notebook 00’s interface and activation–loss distinctions are prerequisites.

## 1. Why this matters: the historical thread

McCulloch and Pitts (1943) described idealized threshold units; Rosenblatt’s perceptron (1957–1958) supplied a learning rule for one linear separator. Analyses of single-layer limits clarified why depth and nonlinearity matter. Multilayer networks became practically trainable when derivatives through compositions could be organized efficiently.

Reverse accumulation appeared in several communities. Rumelhart, Hinton, and Williams (1986) helped popularize backpropagation for learned distributed representations. Modern frameworks automate reverse-mode differentiation at scale, but the mechanism remains organized chain rule.

In [ ]:
from __future__ import annotations

import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from rice_dsm.ml.neural import (
    DenseLayer,
    MeanSquaredNetwork,
    ReLU,
    Sigmoid,
    half_mean_squared_error,
)
from rice_dsm.paths import course_database_path

plt.style.use("seaborn-v0_8-whitegrid")
RANDOM_STATE = 438

## 2. Why do we need neural networks?

A single identity neuron represents an affine response surface. A single sigmoid neuron has affine log-odds and a hyperplane decision boundary. Those restrictions make linear and logistic regression interpretable, but they cannot learn several separated regimes, an XOR interaction, or a hierarchy in which small patterns combine into larger ones unless we first construct suitable features.

A network learns intermediate features jointly with the final prediction:

- **width** lets several neurons detect different directions, thresholds, or regimes;
- **nonlinearity** lets their combination bend instead of collapsing to one affine map; and
- **depth** lets later layers reuse and compose earlier features.

This is a hypothesis, not a guarantee: the response may be simpler as a composition of learned representations than as one function of the original measurements. Networks are especially natural for structured high-dimensional signals, but small tables may still favor linear models, trees, kernels, or domain-designed features.

In [ ]:
# Three views of expressive power: affine collapse, width, and depth.
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))

# A: composing affine functions changes the line, not the function class.
x_grid = np.linspace(-1.0, 1.0, 500)
first_affine = 0.85 * x_grid + 0.18
second_affine = -1.05 * first_affine + 0.12
third_affine = 0.72 * second_affine - 0.08
axes[0].plot(x_grid, first_affine, label="layer 1", linewidth=1.8)
axes[0].plot(x_grid, second_affine, label="layer 2 composition", linewidth=1.8)
axes[0].plot(x_grid, third_affine, label="layer 3 composition", linewidth=2.6)
axes[0].set(title="Affine depth still gives a line", xlabel="x", ylabel="output")
axes[0].legend(fontsize=7)

# B: hidden ReLU units act as hinge features; a weighted sum creates several bends.
def relu(values: np.ndarray) -> np.ndarray:
    return np.maximum(values, 0.0)

hinges = (
    0.75 * relu(x_grid + 0.68),
    -1.55 * relu(x_grid + 0.18),
    1.35 * relu(x_grid - 0.38),
)
combined_features = 0.15 + sum(hinges)
for hinge in hinges:
    axes[1].plot(x_grid, hinge, linewidth=1.2, alpha=0.75)
axes[1].plot(x_grid, combined_features, color="black", linewidth=2.6, label="weighted sum")
axes[1].set(title="Width combines nonlinear features", xlabel="x", ylabel="output")
axes[1].legend(fontsize=7)

# C: one ReLU-representable tent feature is reused through composition.
def tent(values: np.ndarray) -> np.ndarray:
    return 2 * relu(values) - 4 * relu(values - 0.5) + 2 * relu(values - 1.0)

unit_interval = np.linspace(0.0, 1.0, 1001)
composed = unit_interval.copy()
for depth, offset in enumerate((2.3, 1.15, 0.0), start=1):
    composed = tent(composed)
    axes[2].plot(unit_interval, composed + offset, linewidth=2, label=f"depth {depth}")
axes[2].set(title="Depth reuses and composes features", xlabel="x", yticks=[])
axes[2].legend(fontsize=7)

fig.suptitle("Where neural-network expressive power comes from", weight="bold")
plt.tight_layout()
assert np.isfinite(combined_features).all()
assert np.isfinite(composed).all()

## 3. Literal neurons and the $z/a$ distinction

Let $a^0=x$. Weight $w_{j,k}^{\ell}$ connects neuron $k$ in layer $\ell-1$ to neuron $j$ in layer $\ell$:

$$
z_j^\ell=\sum_k w_{j,k}^\ell a_k^{\ell-1}+b_j^\ell,
\qquad
a_j^\ell=\phi^\ell(z_j^\ell).
$$

The superscript identifies the layer; the first subscript identifies the receiving neuron; the second identifies the sending neuron. A bias is a trainable offset, not another observed feature.

In [ ]:
fig, axis = plt.subplots(figsize=(10, 5))
positions = {
    **{f"x{k+1}": (0, 2.5-k) for k in range(3)},
    **{f"h{j+1}": (1, 2-j) for j in range(2)},
    **{f"y{j+1}": (2, 2-j) for j in range(2)},
}
for source in ["x1", "x2", "x3"]:
    for target in ["h1", "h2"]:
        axis.plot([positions[source][0], positions[target][0]],
                  [positions[source][1], positions[target][1]], color="0.75")
for source in ["h1", "h2"]:
    for target in ["y1", "y2"]:
        axis.plot([positions[source][0], positions[target][0]],
                  [positions[source][1], positions[target][1]], color="0.55")
for label, position in positions.items():
    color = "tab:blue" if label.startswith("x") else ("tab:orange" if label.startswith("h") else "tab:green")
    axis.scatter(*position, s=1_800, color=color, edgecolor="black", zorder=2)
    displayed = label if label.startswith("x") else "z, a\n" + label
    axis.text(*position, displayed, ha="center", va="center", color="white", weight="bold")
axis.text(0, 3.0, "a⁰ = x", ha="center")
axis.text(1, 3.0, "hidden layer 1", ha="center")
axis.text(2, 3.0, "output layer 2", ha="center")
axis.set(xlim=(-0.4, 2.4), ylim=(-0.2, 3.3), title="A 3–2–2 network: each circle is one neuron")
axis.axis("off")
plt.tight_layout()

### One observation, neuron by neuron

For the first hidden neuron,

$$
z_1^1=w_{1,1}^1x_1+w_{1,2}^1x_2+w_{1,3}^1x_3+b_1^1,
\qquad a_1^1=\sigma(z_1^1).
$$

An output neuron consumes both hidden activations:

$$
z_1^2=w_{1,1}^2a_1^1+w_{1,2}^2a_2^1+b_1^2,
\qquad a_1^2=\sigma(z_1^2).
$$

Every edge carries a value forward; the same edge will carry sensitivity backward.

In [ ]:
x = np.array([0.8, -0.4, 1.0])
W1 = np.array([[0.2, -0.5, 0.3], [0.7, 0.1, -0.2]])
b1 = np.array([0.1, -0.3])
W2 = np.array([[0.6, -0.4], [-0.2, 0.8]])
b2 = np.array([0.05, -0.10])
sigmoid = Sigmoid()

z1 = W1 @ x + b1
a1 = sigmoid.forward(z1)
z2 = W2 @ a1 + b2
a2 = sigmoid.forward(z2)

display(pd.DataFrame({
    "quantity": ["a^0=x", "z^1", "a^1", "z^2", "a^2=prediction"],
    "value": [x, z1, a1, z2, a2],
}))
assert z1.shape == a1.shape == z2.shape == a2.shape == (2,)

The scalar equations compress to

$$
z^\ell=W^\ell a^{\ell-1}+b^\ell,\qquad a^\ell=\phi^\ell(z^\ell).
$$

For this example $W^1$ has shape $2\times3$, $W^2$ has shape $2\times2$, and each bias, preactivation, and postactivation has length two. Matrix notation evaluates individual neuron equations together; it does not define a different model.

## 4. Nonlinearity creates expressive depth

With identity activations,

$$
W^2(W^1x+b^1)+b^2=(W^2W^1)x+(W^2b^1+b^2),
$$

so any number of layers collapses to one affine map. A nonlinear activation prevents this collapse.

Sigmoid is smooth and useful for the derivation, but $\sigma'(z)$ approaches zero for large $|z|$. ReLU is piecewise linear: $\max(0,z)$. It passes positive signals without sigmoid saturation but requires a derivative convention at zero and can create inactive units.

In [ ]:
grid = np.linspace(-6.0, 6.0, 401)
relu = ReLU()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(grid, sigmoid.forward(grid), label="sigmoid")
axes[0].plot(grid, sigmoid.derivative(grid), "--", label="derivative")
axes[0].set(title="Smooth but saturating", xlabel="z")
axes[0].legend()
axes[1].plot(grid, relu.forward(grid), label="ReLU")
axes[1].plot(grid, relu.derivative(grid), "--", label="chosen derivative")
axes[1].set(title="Piecewise linear", xlabel="z")
axes[1].legend()
plt.tight_layout()

## 5. A transparent first loss and backpropagation

For two sigmoid outputs and a one-hot target, begin with

$$
C=\frac12\sum_{j=1}^{2}(a_j^2-y_j)^2.
$$

Define the local error with respect to preactivation:

$$
\delta_j^2:=\frac{\partial C}{\partial z_j^2}
=(a_j^2-y_j)\sigma'(z_j^2).
$$

For an output weight, follow one path:

$$
\frac{\partial C}{\partial w_{j,k}^2}
=\frac{\partial C}{\partial a_j^2}
 \frac{\partial a_j^2}{\partial z_j^2}
 \frac{\partial z_j^2}{\partial w_{j,k}^2}
=\delta_j^2a_k^1.
$$

Also $\partial C/\partial b_j^2=\delta_j^2$. The forward pass retains $a^{\ell-1}$ and $z^\ell$ because the backward pass needs them.

### Hidden errors sum downstream paths

Hidden neuron $j$ affects every output neuron, so its paths add:

$$
\delta_j^1=
\left(\sum_r w_{r,j}^2\delta_r^2\right)\sigma'(z_j^1).
$$

In vector form,

$$
\boxed{\delta^\ell=((W^{\ell+1})^\top\delta^{\ell+1})\odot\phi'(z^\ell)}
$$

and

$$
\boxed{\frac{\partial C}{\partial W^\ell}
=\delta^\ell(a^{\ell-1})^\top,\qquad
\frac{\partial C}{\partial b^\ell}=\delta^\ell.}
$$

Backpropagation is dynamic programming for the chain rule: a downstream sensitivity is computed once and reused.

In [ ]:
target = np.array([1.0, 0.0])
delta2 = (a2 - target) * sigmoid.derivative(z2)
gradient_W2 = np.outer(delta2, a1)
delta1 = (W2.T @ delta2) * sigmoid.derivative(z1)
gradient_W1 = np.outer(delta1, x)

for name, value in {
    "delta^2": delta2,
    "dC/dW^2": gradient_W2,
    "delta^1": delta1,
    "dC/dW^1": gradient_W1,
}.items():
    print(f"{name:8s} shape={value.shape}\n{value}\n")
assert gradient_W1.shape == W1.shape
assert gradient_W2.shape == W2.shape

## 6. Literal neurons become a vectorized layer

For a batch $A^{\ell-1}\in\mathbb R^{n\times d_{\ell-1}}$,

$$
Z^\ell=A^{\ell-1}(W^\ell)^\top+b^\ell,\qquad A^\ell=\phi(Z^\ell),
$$

and

$$
\nabla_{W^\ell}C=(\Delta^\ell)^\top A^{\ell-1}.
$$

DenseLayer is a vectorized collection of the same neurons. Its forward method caches values, backward consumes an upstream gradient, and step updates only after all gradients are known.

In [ ]:
X_xor = np.array([
    [0.0, 0.0, 1.0],
    [0.0, 1.0, 1.0],
    [1.0, 0.0, 1.0],
    [1.0, 1.0, 1.0],
])
y_xor = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [0.0, 1.0],
    [1.0, 0.0],
])
network = MeanSquaredNetwork([
    DenseLayer(3, 2, Sigmoid(), random_state=1),
    DenseLayer(2, 2, Sigmoid(), random_state=2),
])
loss_history = [
    network.train_step(X_xor, y_xor, learning_rate=1.0)
    for _ in range(5_000)
]
outputs = network.forward(X_xor)
display(pd.DataFrame({
    "x1": X_xor[:, 0],
    "x2": X_xor[:, 1],
    "target": y_xor.argmax(axis=1),
    "prediction": outputs.argmax(axis=1),
    "output 0": outputs[:, 0],
    "output 1": outputs[:, 1],
}))
assert loss_history[-1] < loss_history[0]
assert np.isfinite(outputs).all()

In [ ]:
fig, axis = plt.subplots(figsize=(8, 4))
axis.plot(loss_history, color="tab:purple")
axis.set(title="Optimization evidence for the 3–2–2 network",
         xlabel="full-batch update", ylabel="half MSE", yscale="log")
plt.tight_layout()

## 7. Debugging with gradient checks and common failure modes

A transpose or broadcasting bug can still produce a falling curve. Perturb each parameter on a tiny deterministic problem and compare the central difference with the analytical gradient. Use double precision and a smooth, nonsaturated example. This is a development test, not a production optimizer.

In [ ]:
layer = DenseLayer(2, 2, Sigmoid(), random_state=4)
check_X = np.array([[0.2, -0.4], [1.1, 0.3]])
check_y = np.array([[1.0, 0.0], [0.0, 1.0]])
prediction = layer.forward(check_X)
layer.backward((prediction - check_y) / len(check_X))
analytical = layer.weight_gradient.copy()

numerical = np.zeros_like(layer.weights)
step = 1e-6
for index in np.ndindex(layer.weights.shape):
    original = layer.weights[index]
    layer.weights[index] = original + step
    plus = half_mean_squared_error(layer.forward(check_X), check_y)
    layer.weights[index] = original - step
    minus = half_mean_squared_error(layer.forward(check_X), check_y)
    layer.weights[index] = original
    numerical[index] = (plus - minus) / (2 * step)

print("largest discrepancy:", np.max(np.abs(analytical - numerical)))
np.testing.assert_allclose(analytical, numerical, rtol=1e-5, atol=1e-7)

## 8. Modern output contracts

The transparent sigmoid–MSE example gives

$$
\delta^L=(a^L-y)\odot\sigma'(z^L).
$$

Binary sigmoid with cross-entropy simplifies to

$$
\delta^L=a^L-y.
$$

For mutually exclusive multiclass labels, modern code usually supplies raw logits to one stable softmax-cross-entropy operation. Hidden recursion remains unchanged. The loss changes the initial backward signal; it does not replace backpropagation.

Initialization must also match activation behavior. Xavier/Glorot scaling is motivated by signal variance for symmetric activations; He scaling is commonly paired with ReLU. These are starting rules, not guarantees. Record the initializer, seed, dtype, architecture, and activation.

## 9. Worked example: real diagnostic measurements

We apply the manual network to the Wisconsin breast-cancer table. This is a teaching experiment, not a clinical device. We preserve an untouched test partition, fit scaling only on training observations, and show balanced accuracy plus a confusion matrix because false negatives and false positives have distinct meanings.

In [ ]:
database_uri = f"{course_database_path().as_uri()}?mode=ro"
with sqlite3.connect(database_uri, uri=True) as connection:
    cancer = pd.read_sql_query(
        """
        SELECT mean_radius, mean_texture, mean_smoothness,
               mean_concavity, mean_concave_points, diagnosis_code
        FROM breast_cancer_observations
        """,
        connection,
    )

X = cancer.drop(columns="diagnosis_code")
y = cancer["diagnosis_code"].to_numpy(dtype=int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)
y_train_one_hot = np.eye(2)[y_train]

print("training shape:", X_train.shape)
print("test shape:", X_test.shape)
display(cancer.groupby("diagnosis_code").size().rename("count"))

In [ ]:
cancer_network = MeanSquaredNetwork([
    DenseLayer(X_train.shape[1], 8, Sigmoid(), random_state=10),
    DenseLayer(8, 2, Sigmoid(), random_state=11),
])
training_loss = [
    cancer_network.train_step(X_train, y_train_one_hot, learning_rate=0.5)
    for _ in range(1_500)
]
test_outputs = cancer_network.forward(X_test)
test_predictions = test_outputs.argmax(axis=1)
score = balanced_accuracy_score(y_test, test_predictions)
print(f"held-out balanced accuracy: {score:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(training_loss)
axes[0].set(title="Training objective", xlabel="update", ylabel="half MSE", yscale="log")
ConfusionMatrixDisplay.from_predictions(
    y_test, test_predictions, display_labels=["class 0", "class 1"],
    ax=axes[1], colorbar=False
)
axes[1].set_title("Held-out decisions")
plt.tight_layout()

The loss trace shows whether the optimizer reduced its training objective. The confusion matrix shows held-out decisions. Neither proves calibrated probabilities: two sigmoid outputs need not sum to one, and MSE was chosen to mirror the derivation.

A modern follow-up uses logits-based cross-entropy, validation-based model selection, uncertainty estimates, and an explicit decision policy. The manual network is a microscope for the mechanism before a framework owns derivatives.

## 10. What automatic differentiation changes

An autodiff framework records tensor operations and applies local derivative rules in reverse. It enables optimized batching, accelerators, and large architectures. It does not decide whether the target is valid, the split leaks, the loss matches the claim, the metric matches operational cost, or saved preprocessing matches inference.

Monitor training and validation loss separately, gradient norms, nonfinite values, throughput, and resource use. A decreasing loss is necessary optimization evidence, never sufficient release evidence.

## Professional practice and practice ladder

| Data scientist asks | Software engineer asks |
|---|---|
| What representation may improve prediction? | Are tensor shapes explicit? |
| Does output/loss match the target? | Does loss consume logits or activations? |
| Is improvement real on held-out data? | Can the checkpoint reload and reproduce it? |
| Are errors consequential for a subgroup? | Are class order and metrics versioned? |

**Guided:** derive $\partial C/\partial w_{1,2}^1$ by listing both output paths.

**Independent:** replace hidden sigmoid with ReLU and justify initialization.

**Extension:** implement stable softmax cross-entropy consuming logits; test extreme scores and gradients.

**Contribution:** add minibatching, early stopping, serialization, an activation, or a logits loss with shape contracts, reproducible tests, and gradient evidence.

## Retrieval practice, takeaway, and further reading

1. How do $z_j^\ell$ and $a_j^\ell$ differ?
2. Why do downstream paths add in a hidden error?
3. Where does the transpose in $(W^{\ell+1})^\top\delta^{\ell+1}$ come from?
4. Why does cross-entropy change the sigmoid output delta?
5. Which tasks should autodiff replace, and which judgments remain ours?

**Takeaway:** a dense layer is a vectorized collection of neurons, backpropagation is organized chain rule, and trustworthy implementation connects derivation, shape contracts, gradient checks, held-out evidence, and reproducible software.

**Further reading:** [Python data model](https://docs.python.org/3/reference/datamodel.html), NumPy documentation, and the cited primary neural-network literature.